# CK ck extraction

In [1]:
from collections import defaultdict
from datetime import datetime
import os
import time
import pickle
import pandas as pd
import json as json
import numpy as np
import itertools

def get_cf_dict(datasets_name, data_type):
    data_path = 'datasets/' + datasets_name + '/' + data_type + '.pkl'

    print("start time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

    with open(data_path, 'rb') as f:
        cf_dict = pickle.load(f)


    # train data，analysis

    list_length = list(map(lambda x: list(x.values()), list(cf_dict.values())))
#     flat_length = list(np.array(list_length).flatten())
    flat_length = list(itertools.chain.from_iterable(list_length))
    cf_pd = pd.DataFrame({'source':flat_length, 'length':flat_length})

    type_count = pd.DataFrame(cf_pd.groupby(cf_pd['length']).count())
    type_count = type_count.reset_index()[['source', 'length']]
    type_count=type_count.rename(columns={'source':'count','length':'cf_relations'})[['cf_relations','count']]

    type_count['count'] = type_count['count']/2
    type_count['count'] = type_count['count'].apply(int)

    print("end time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
#     print("Type: ",data_type)
    print("distribution of data in",data_type," : ")
    print(type_count[type_count['cf_relations'] != 0].to_string(index=False))
    return cf_dict

 



def get_target_infor(cf_dict, name_t):
    # target distribution
    # train/test
    def get_cf(seq_x,lab_y):
        u_seq = np.unique(seq_x)
        cf_list = []
        for x in u_seq:
            if x != lab_y:
                try:
                    relatios_t = cf_dict[lab_y][x]
                except:
                    relatios_t = -1
                cf_list.append(relatios_t)
        return cf_list
    
    def dedu(s):
        res = list(np.unique(s))
        if len(res) > 1 and -1 in res:
            r_list = [x for x in res if x != -1]
        else:
            r_list = res
        return r_list
    
    target_path = 'datasets/' + datasets_name + '/' + name_t + '.txt'

#     print("start time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

    print("data type:",name_t)

    with open(target_path, 'rb') as f:
        seqs, labels = pickle.load(f)

    seqs_num = len(labels)
    sl_dict = {'sequence':seqs, 'label':labels}
    sl_pds = pd.DataFrame(sl_dict)    

    sl_pds['cf_infor'] = sl_pds.apply(lambda row: get_cf(row['sequence'], row['label']), axis=1)

    sl_pds['cf_yi'] = sl_pds['cf_infor'].map(dedu)
    
    lab_cf = sl_pds['cf_yi'].tolist()
    flat_length = list(itertools.chain.from_iterable(lab_cf))
    cf_pd = pd.DataFrame({'source':flat_length, 'cf_in':flat_length})

    cf_count = pd.DataFrame(cf_pd.groupby(cf_pd['cf_in']).count())
    cf_count = cf_count.reset_index()[['source', 'cf_in']]
    cf_count=cf_count.rename(columns={'source':'count'})[['cf_in','count']]



    cf_count['frequency'] = cf_count['count']/seqs_num

#     print("end time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
    print("mixed cf infor")
    print(cf_count.to_string(index=False))
    
    
    sl_pds['num'] = sl_pds['cf_yi'].map(len)
    tar_single = sl_pds[sl_pds['num'] == 1]

    lab_cf_yi = tar_single['cf_yi'].tolist()
    flat_length_yi = list(itertools.chain.from_iterable(lab_cf_yi))
    cf_pd_yi = pd.DataFrame({'source_yi':flat_length_yi, 'cf_in_yi':flat_length_yi})

    cf_count_yi = pd.DataFrame(cf_pd_yi.groupby(cf_pd_yi['cf_in_yi']).count())
    cf_count_yi = cf_count_yi.reset_index()[['source_yi', 'cf_in_yi']]
    cf_count_yi = cf_count_yi.rename(columns={'source_yi':'count_yi'})[['cf_in_yi','count_yi']]

    cf_count_yi['frequency_yi'] = cf_count_yi['count_yi']/seqs_num
    print("single cf infor")
    print(cf_count_yi.to_string(index=False))
    
    return sl_pds

/home/dutir923/zhangxiaokun/anaconda3/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)


In [2]:

datasets_name = 'Tmall'

# all_train/all_test/all_seqs 
dict_res = 'all_train'
train_cf_dict = get_cf_dict(datasets_name, dict_res)

# train/test
if dict_res == 'all_train':
    tar_name = "test"
    test_target = get_target_infor(train_cf_dict, tar_name)

start time:  2024-12-17 17:46:39
end time:  2024-12-17 17:49:00
distribution of data in all_train  : 
 cf_relations    count
            1   318964
            2 11299374
            3 77262723
            4 53084498
            5  7272328
            6   596621
            7    42634
            8     1484
            9       49
           10        2
data type: test
mixed cf infor
 cf_in  count  frequency
     1    796   0.384727
     2    895   0.432576
     3   1147   0.554374
     4    571   0.275979
     5    116   0.056066
     6     18   0.008700
single cf infor
 cf_in_yi  count_yi  frequency_yi
        1       237      0.114548
        2       205      0.099082
        3       364      0.175930
        4       163      0.078782
        5        28      0.013533
        6         5      0.002417


## CR distribution about Label! train [x1,x2] -> [x3]

In [5]:
from collections import defaultdict
from datetime import datetime
import os
import time
import pickle
import pandas as pd
import json as json
import numpy as np
import itertools



datasets_name = 'Tmall'

print("start time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
# train
data_type = 'all_train'
cr_record_path = 'datasets/' + datasets_name + '/' + data_type + '.pkl'

with open(cr_record_path, 'rb') as f:
    cf_dict = pickle.load(f)

test_data_path = 'datasets/' + datasets_name + '/test.txt'

with open(test_data_path, 'rb') as f:
    seqs, labels = pickle.load(f)
    

num_none = 0
num_1 = 0
num_2 = 0
num_3 = 0
num_4 = 0
num_5 = 0
num_x = 0
for x_lab, y_seq in zip(labels, seqs):
#     y_seq_uni = np.unique(y_seq).tolist()
    for y_item in y_seq:
        try:
            cr_temp = cf_dict[x_lab][y_item]
        except:
            cr_temp = -1
        if cr_temp == -1:
            num_none += 1
        elif cr_temp == 1:
            num_1 += 1
        elif cr_temp == 2:
            num_2 += 1
        elif cr_temp == 3:
            num_3 += 1
        elif cr_temp == 4:
            num_4 += 1
        elif cr_temp == 5:
            num_5 += 1
        else:
            num_x += 1
print('dataset name:',datasets_name)
print('CR distribution about Label!')
print('#none-hop: ',str(num_none))
print('#0-hop: ',str(num_1))
print('#1-hop: ',str(num_2))
print('#2-hop: ',str(num_3))
print('#3-hop: ',str(num_4))
print('#4-hop: ',str(num_5))
print('#others: ',str(num_x))

print("end time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

start time:  2024-12-17 18:14:30
dataset name: Tmall
CR distribution about Label!
#none-hop:  15
#0-hop:  2095
#1-hop:  3177
#2-hop:  4299
#3-hop:  1466
#4-hop:  299
#others:  2278
end time:  2024-12-17 18:14:49


## Proportion of Different CRs in Predictions

In [1]:
from collections import defaultdict
from datetime import datetime
import os
import time
import pickle
import pandas as pd
import json as json
import numpy as np
import itertools
from collections import Counter


def get_pre_ck(x_seq, y_pre,cf_dict, pre_num):
#     seq_arr = np.unique(x_seq)
    seq_arr = np.array(x_seq)
    seqs_list = seq_arr[np.nonzero(seq_arr)[0]].tolist()
    u_pre_list = y_pre[:pre_num]
    ck_all = []
    for x_pre in u_pre_list:
        try:
            x_pre = int(x_pre)
        except:
            x_pre = -100
        
        ck_in = []
        for i_seq in seqs_list:
            try:
                ck_pre_seq = cf_dict[i_seq][x_pre]
            except:
#                     print('none CK between ',str(i_seq),' and ',str(x_pre))
                ck_pre_seq = -1
#                 if ck_pre_seq not in ck_in:
#                     ck_in.append(ck_pre_seq)
            ck_in.append(ck_pre_seq)
#         if len(ck_in) == 0:
#             ck_in.append(-1)
        ck_all += ck_in
    return ck_all

def ck_patterns(datasets_name,model_n,top_number):
    

#     print("Start time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

    data_path = 'datasets/' + datasets_name + '/all_train.pkl'
    with open(data_path, 'rb') as f:
        cf_dict = pickle.load(f)

    
    pre_path = 'ck_patterns/' + model_n + '/' + datasets_name + '/prediction.txt'
    if model_n == "P5":
        pre_path = 'ck_patterns/' + model_n + '/' + datasets_name + '/prediction.pkl'

    with open(pre_path, 'rb') as f:
        seqs, labels, pred = pickle.load(f)

    seqs_num = len(labels)
    slp_dict = {'sequence':seqs, 'prediction':pred, 'label':labels}
    slp_pds = pd.DataFrame(slp_dict)    

    slp_pds['ck_relation'] = slp_pds.apply(lambda row: get_pre_ck(row['sequence'], row['prediction'], cf_dict, top_number), axis=1)
    ck_all= list(itertools.chain.from_iterable(slp_pds['ck_relation'].tolist()))
    
#     print(slp_pds.head(10))

    ck_fre_dict = Counter(ck_all)
    ck_fre_dict = dict(sorted(ck_fre_dict.items()))
    fenmu = len(ck_all)
    print(datasets_name)
    print("top-",str(top_number))
    for ck_key in ck_fre_dict:
        fre_value = ck_fre_dict[ck_key]
        res = fre_value/fenmu
        print("ck type:",str(ck_key)," frequence: ",str(fre_value), " frequency",str(res))

#     print("End time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
#     print("Over")

/home/dutir923/zhangxiaokun/anaconda3/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)


In [ ]:
# Grocery_and_Gourmet_Food/Cell_Phones_and_Accessories/cosmetics/diginetica/yoochoose/Instacart
# datasets_list = ['Grocery_and_Gourmet_Food','Cell_Phones_and_Accessories','cosmetics','diginetica','yoochoose','Tmall']
datasets_list = ['Tmall']
# NARM, SKNN SRGNN BERT4Rec DHCN, P5
model_list = ['SKNN','NARM','BERT4Rec','DHCN','P5']

# [10, 5, 1]
cut_list = [10]

for model_name in model_list:
    print("model: ",model_name)
    for datasets in datasets_list:
        for cutoff in cut_list:
            ck_patterns(datasets,model_name,cutoff)

model:  SKNN
Tmall
top- 10
ck type: -1  frequence:  195  frequency 0.0014394755879704132
ck type: 0  frequence:  9538  frequency 0.0704088110669836
ck type: 1  frequence:  44479  frequency 0.3283406906530052
ck type: 2  frequence:  33996  frequency 0.25095595942893417
ck type: 3  frequence:  35026  frequency 0.2585593433038548
ck type: 4  frequence:  10480  frequency 0.07736258544579451
ck type: 5  frequence:  1568  frequency 0.011574860112500553
ck type: 6  frequence:  177  frequency 0.0013066009183116058
ck type: 7  frequence:  7  frequency 5.167348264509176e-05
model:  NARM
Tmall
top- 10
ck type: -1  frequence:  487  frequency 0.0035732628952967935
ck type: 0  frequence:  5308  frequency 0.03894636437009318
ck type: 1  frequence:  6558  frequency 0.04811798371120405
ck type: 2  frequence:  19701  frequency 0.14455205811138014
ck type: 3  frequence:  67589  frequency 0.4959204637170739
ck type: 4  frequence:  32493  frequency 0.2384107418005723
ck type: 5  frequence:  3812  frequency

In [5]:
len(list(itertools.chain.from_iterable(s)))

7

## Item co-occurrence frequency in CRs，zero-hop frequency

In [1]:
from collections import defaultdict
from datetime import datetime
import os
import time
import pickle
import pandas as pd
import json as json
import numpy as np
import itertools
from collections import Counter

# Grocery_and_Gourmet_Food/Cell_Phones_and_Accessories/cosmetics/diginetica/yoochoose/Instacart/Tmall

datasets_name = 'Tmall'
data_type = 'all_train'
data_path = 'datasets/' + datasets_name + '/' + data_type + '.txt'

print("start time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

if datasets_name == 'Cell_Phones_and_Accessories':
    n_items = 10737
elif datasets_name == 'Grocery_and_Gourmet_Food':
    n_items = 7910
elif datasets_name == 'cosmetics':
    n_items = 24100
elif datasets_name == 'diginetica':
    n_items = 22507
elif datasets_name == 'yoochoose':
    n_items = 11642
elif datasets_name == 'Instacart':
    n_items = 14708
elif datasets_name == 'Tmall':
    n_items = 17360

with open(data_path, 'rb') as f:
    seqs= pickle.load(f)
    
print("data name:",datasets_name)
print("data type:",data_type)


fre_zero_matrix = np.zeros((n_items+1, n_items+1))
for t_seq in seqs:
    t_seq = np.unique(t_seq)
    for x_idx in range(len(t_seq)-1):
        for y_idx in range(x_idx+1, len(t_seq)):
            pre_item = t_seq[x_idx]
            tai_item = t_seq[y_idx]
            fre_zero_matrix[pre_item][tai_item] += 1
mat_list = list(itertools.chain.from_iterable(fre_zero_matrix.tolist()))

fre_dict = Counter(mat_list)
ck_fre_dict = dict(sorted(fre_dict.items()))


print("**************"*5)
print("co-occurrent times\t","#item pairs")
for ck_key in ck_fre_dict:
    fre_value = ck_fre_dict[ck_key]
    print(str(int(ck_key)),"\t ",str(fre_value))
    
print("End time: ",time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

/home/dutir923/zhangxiaokun/anaconda3/lib/python3.7/site-packages/pandas/compat/_optional.py:138: UserWarning: Pandas requires version '2.7.0' or newer of 'numexpr' (version '2.6.9' currently installed).
  warnings.warn(msg, UserWarning)


start time:  2024-12-17 19:41:29
data name: Tmall
data type: all_train
**********************************************************************
co-occurrent times	 #item pairs
0 	  301085357
1 	  284860
2 	  25318
3 	  5938
4 	  1734
5 	  600
6 	  251
7 	  102
8 	  55
9 	  35
10 	  22
11 	  12
12 	  7
13 	  6
14 	  4
15 	  1
16 	  6
17 	  2
19 	  1
20 	  1
21 	  1
22 	  1
25 	  1
26 	  1
30 	  1
35 	  1
37 	  1
38 	  1
63 	  1
End time:  2024-12-17 19:42:03
